In [21]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from scipy.signal import butter, filtfilt
import mediapipe as mp

class TwoStageRPPGDataset(Dataset):
    def __init__(self, csv_path, data_dir, patient_list=None, target_frames=600, fps=30.0, global_bounds=None):
        """
        Stage 1 Dataset: Type-safe patient isolation mapping.
        """
        self.df = pd.read_csv(csv_path)
        self.data_dir = data_dir
        self.target_frames = target_frames
        self.fps = fps
        self.camera_list = sorted(self.df['camera'].unique().tolist())
        
        if global_bounds:
            self.age_min, self.age_max = global_bounds['age_min'], global_bounds['age_max']
            self.bmi_min, self.bmi_max = global_bounds['bmi_min'], global_bounds['bmi_max']
        else:
            self.age_min, self.age_max = self.df['age'].min(), self.df['age'].max()
            self.bmi_min, self.bmi_max = self.df['bmi'].min(), self.df['bmi'].max()

        self.FACE_LANDMARKS = [10, 338, 297, 332, 284, 251, 21, 54, 103, 67, 109]

        # --- FIX: Safe Type Casting for Comparison ---
        allowed_patients = None
        if patient_list is not None:
            # Force everything to strings to prevent string vs int comparison failures
            allowed_patients = set(str(p).strip() for p in patient_list)

        self.valid_rows = []
        missing_file_count = 0
        skipped_patient_count = 0
        
        for idx, row in self.df.iterrows():
            current_row_patient = str(row['patient_id']).strip()
            
            # Check patient membership safely
            if allowed_patients is not None and current_row_patient not in allowed_patients:
                skipped_patient_count += 1
                continue
                
            video_path = os.path.join(self.data_dir, str(row['video']).strip())
            if os.path.exists(video_path):
                self.valid_rows.append(row)
            else:
                missing_file_count += 1
                
        print(f"📦 Stage 1 Pipeline initialized:")
        print(f"   ↳ {len(self.valid_rows)} samples successfully matched.")
        if skipped_patient_count > 0:
            print(f"   ↳ {skipped_patient_count} rows filtered out (allocated to different split).")
        if missing_file_count > 0:
            print(f"   ⚠️ Warning: {missing_file_count} rows skipped because video file wasn't found at: {self.data_dir}")

    def __len__(self):
        return len(self.valid_rows)

    def _butter_bandpass_filter(self, data, lowcut=0.75, highcut=3.0, order=3):
        nyq = 0.5 * self.fps
        low = lowcut / nyq
        high = highcut / nyq
        b, a = butter(order, [low, high], btype='band')
        return filtfilt(b, a, data)

    def _apply_chrom_rppg(self, rgb_traces):
        X = np.array(rgb_traces, dtype=np.float32)
        X_mean = np.mean(X, axis=0, keepdims=True)
        X_norm = X / (X_mean + 1e-6)
        
        S1 = 3.0 * X_norm[:, 0] - 2.0 * X_norm[:, 1]       
        S2 = 1.5 * X_norm[:, 0] + 1.0 * X_norm[:, 1] - 1.5 * X_norm[:, 2]  
        
        alpha = np.std(S1) / (np.std(S2) + 1e-6)
        bvp_signal = S1 - (alpha * S2)
        
        try:
            bvp_signal = self._butter_bandpass_filter(bvp_signal)
        except Exception:
            pass
            
        std = np.std(bvp_signal)
        if std > 1e-6:
            bvp_signal = (bvp_signal - np.mean(bvp_signal)) / std
            
        return bvp_signal

    def _extract_dynamic_face_signal(self, video_path):
        cap = cv2.VideoCapture(video_path)
        rgb_traces = []
        frame_count = 0
        
        mp_face_mesh = mp.solutions.face_mesh
        
        with mp_face_mesh.FaceMesh(
            static_image_mode=False, 
            max_num_faces=1, 
            refine_landmarks=False, 
            min_detection_confidence=0.5
        ) as face_mesh:
            
            while cap.isOpened() and frame_count < self.target_frames:
                ret, frame = cap.read()
                if not ret:
                    break
                
                h, w, _ = frame.shape
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = face_mesh.process(frame_rgb)
                
                if results.multi_face_landmarks:
                    landmarks = results.multi_face_landmarks[0].landmark
                    mask = np.zeros((h, w), dtype=np.uint8)
                    
                    pts = np.array([(int(landmarks[idx].x * w), int(landmarks[idx].y * h)) for idx in self.FACE_LANDMARKS])
                    cv2.fillConvexPoly(mask, pts, 255)
                    
                    mean_val = cv2.mean(frame_rgb, mask=mask)[:3]
                    rgb_traces.append(mean_val)
                else:
                    mean_val = cv2.mean(frame_rgb)[:3]
                    rgb_traces.append(mean_val)
                    
                frame_count += 1
                
        cap.release()
        
        if len(rgb_traces) == 0:
            rgb_traces = [[0.0, 0.0, 0.0]]
        while len(rgb_traces) < self.target_frames:
            rgb_traces.append(rgb_traces[-1])
            
        clean_1d_wave = self._apply_chrom_rppg(rgb_traces)
        return np.expand_dims(clean_1d_wave, axis=0).astype(np.float32)

    def __getitem__(self, idx):
        row = self.valid_rows[idx]
        video_full_path = os.path.join(self.data_dir, row['video'])
        
        x_signal_tensor = torch.from_numpy(self._extract_dynamic_face_signal(video_full_path))
        
        norm_age = (float(row['age']) - self.age_min) / (self.age_max - self.age_min + 1e-5)
        norm_bmi = (float(row['bmi']) - self.bmi_min) / (self.bmi_max - self.bmi_min + 1e-5)
        is_male = 1.0 if str(row['sex']).strip().upper() == 'M' else 0.0
        
        step_str = str(row['step']).strip().lower()
        is_before = 1.0 if step_str == 'before' else 0.0
        is_after  = 1.0 if step_str == 'after' else 0.0
        is_rest   = 1.0 if step_str == 'rest' else 0.0
        
        current_cam = str(row['camera']).strip()
        cam_context = [1.0 if current_cam == cam else 0.0 for cam in self.camera_list]
        
        x_context = [norm_age, norm_bmi, is_male, is_before, is_after, is_rest] + cam_context
        x_context_tensor = torch.tensor(x_context, dtype=torch.float32)
        
        targets = {
            'pulse': torch.tensor(float(row['pulse']), dtype=torch.float32),
            'saturation': torch.tensor(float(row['saturation']), dtype=torch.float32),
            'upper_ap': torch.tensor(float(row['upper_ap']), dtype=torch.float32),
            'lower_ap': torch.tensor(float(row['lower_ap']), dtype=torch.float32),
            'glycated_hemoglobin': torch.tensor(float(row['glycated_hemoglobin']), dtype=torch.float32),
            'hemoglobin': torch.tensor(float(row['hemoglobin']), dtype=torch.float32),
            'cholesterol': torch.tensor(float(row['cholesterol']), dtype=torch.float32),
            'temperature': torch.tensor(float(row['temperature']), dtype=torch.float32),
            'respiratory': torch.tensor(float(row['respiratory']), dtype=torch.float32)
        }
        
        return x_signal_tensor, x_context_tensor, targets

In [22]:
import torch
import torch.nn as nn

class WaveformMultiTaskRPPGNet(nn.Module):
    def __init__(self, context_dim=9):
        super(WaveformMultiTaskRPPGNet, self).__init__()
        
        # Modified backbone: in_channels changed to 1 because Stage 1 handles the color decoding!
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=9, stride=2, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) 
        )
        
        combined_dim = 128 + context_dim
        
        self.shared_dense = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Specialized Regression Heads
        self.head_pulse               = nn.Linear(128, 1)
        self.head_saturation          = nn.Linear(128, 1)
        self.head_upper_ap            = nn.Linear(128, 1)
        self.head_lower_ap            = nn.Linear(128, 1)
        self.head_glycated_hemoglobin = nn.Linear(128, 1)
        self.head_hemoglobin          = nn.Linear(128, 1)
        self.head_cholesterol         = nn.Linear(128, 1)
        self.head_temperature         = nn.Linear(128, 1)
        self.head_respiratory         = nn.Linear(128, 1)

    def forward(self, signal, context):
        features = self.feature_extractor(signal)
        features = features.view(features.size(0), -1) 
        
        fused = torch.cat((features, context), dim=1)
        shared_out = self.shared_dense(fused)
        
        return {
            'pulse': self.head_pulse(shared_out).squeeze(-1),
            'saturation': self.head_saturation(shared_out).squeeze(-1),
            'upper_ap': self.head_upper_ap(shared_out).squeeze(-1),
            'lower_ap': self.head_lower_ap(shared_out).squeeze(-1),
            'glycated_hemoglobin': self.head_glycated_hemoglobin(shared_out).squeeze(-1),
            'hemoglobin': self.head_hemoglobin(shared_out).squeeze(-1),
            'cholesterol': self.head_cholesterol(shared_out).squeeze(-1),
            'temperature': self.head_temperature(shared_out).squeeze(-1),
            'respiratory': self.head_respiratory(shared_out).squeeze(-1)
        }

print("✅ Waveform Multi-Task Net declared successfully.")

✅ Waveform Multi-Task Net declared successfully.


In [24]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# 1. CONFIGURATION PATHS
csv_file = "D:/project/mcd_rppg_60_patients/db.csv"
data_directory = "D:/project/mcd_rppg_60_patients"

# 2. PATIENT SPLIT LOGIC
df_master = pd.read_csv(csv_file)
unique_patients = df_master['patient_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_patients)

split_idx = int(len(unique_patients) * 0.8)
train_pts = unique_patients[:split_idx]
val_pts = unique_patients[split_idx:]

df_train_leak_free = df_master[df_master['patient_id'].isin(train_pts)]
train_bounds = {
    'age_min': df_train_leak_free['age'].min(), 'age_max': df_train_leak_free['age'].max(),
    'bmi_min': df_train_leak_free['bmi'].min(), 'bmi_max': df_train_leak_free['bmi'].max()
}

# 3. INITIALIZE STAGE 1 LOGIC DATASETS
print("Initializing Stage 1 Augmented Train Dataset (MediaPipe + CHROM)...")
train_dataset = TwoStageRPPGDataset(csv_path=csv_file, data_dir=data_directory, patient_list=train_pts, global_bounds=train_bounds)

print("\nInitializing Stage 1 Augmented Validation Dataset...")
val_dataset = TwoStageRPPGDataset(csv_path=csv_file, data_dir=data_directory, patient_list=val_pts, global_bounds=train_bounds)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# 4. INSTANTIATE STAGE 2 NETWORK
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WaveformMultiTaskRPPGNet(context_dim=9).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.002, weight_decay=0.02)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)
criterion = nn.MSELoss()

loss_weights = {
    'pulse': 0.15,         # Increased slightly to push optimization focus
    'saturation': 1.0,
    'upper_ap': 0.15,      # Increased slightly
    'lower_ap': 0.1,
    'glycated_hemoglobin': 1.0,
    'hemoglobin': 0.5,
    'cholesterol': 0.5,
    'temperature': 1.0,
    'respiratory': 0.5
}

num_epochs = 20
print(f"\n🚀 Running Two-Stage Pipeline Optimization Engine (20 Epochs) on: {device}")
print("="*85)

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    for signals, contexts, targets in train_loader:
        signals, contexts = signals.to(device), contexts.to(device)
        optimizer.zero_grad()
        predictions = model(signals, contexts)
        
        total_loss = 0.0
        for name in loss_weights.keys():
            raw_loss = criterion(predictions[name], targets[name].to(device))
            total_loss += loss_weights[name] * raw_loss
            
        total_loss.backward()
        optimizer.step()
        running_train_loss += total_loss.item()
        
    # VALIDATION CRITERIA TRACKING
    model.eval()
    running_val_loss = 0.0
    val_targets_accum = {key: [] for key in loss_weights.keys()}
    val_preds_accum = {key: [] for key in loss_weights.keys()}
    
    with torch.no_grad():
        for signals, contexts, targets in val_loader:
            signals, contexts = signals.to(device), contexts.to(device)
            predictions = model(signals, contexts)
            
            total_loss = 0.0
            for name in loss_weights.keys():
                raw_loss = criterion(predictions[name], targets[name].to(device))
                total_loss += loss_weights[name] * raw_loss
                val_targets_accum[name].append(targets[name].cpu().numpy())
                val_preds_accum[name].append(predictions[name].cpu().numpy())
                
            running_val_loss += total_loss.item()
            
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    epoch_train = running_train_loss / len(train_loader)
    epoch_val = running_val_loss / len(val_loader)
    
    # Track metrics output structure cleanly at milestones
    if (epoch + 1) == 1 or (epoch + 1) == 20 or (epoch + 1) % 4 == 0:
        print(f"📈 Epoch [{epoch+1}/{num_epochs}] | LR: {current_lr:.5f} | Train Loss: {epoch_train:.4f} | Weighted Val Loss: {epoch_val:.4f}")
        print("="*85)
        print(f"{'Task / Vital Sign':<22} | {'MAE (↓)':<10} | {'RMSE (↓)':<10} | {'MAPE % (↓)':<10}")
        print("-"*85)
        
        for name in loss_weights.keys():
            y_true = np.concatenate(val_targets_accum[name]).flatten()
            y_pred = np.concatenate(val_preds_accum[name]).flatten()
            
            mae = mean_absolute_error(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mape = mean_absolute_percentage_error(y_true, y_pred) * 100
            
            print(f" • {name.ljust(20)} | {mae:<10.3f} | {rmse:<10.3f} | {mape:<10.2f}%")
        print("="*85 + "\n")

print("🎉 Two-Stage Modular System Training Complete!")

Initializing Stage 1 Augmented Train Dataset (MediaPipe + CHROM)...
📦 Stage 1 Pipeline initialized:
   ↳ 288 samples successfully matched.
   ↳ 720 rows filtered out (allocated to different split).
   ⚠️ Warning: 2592 rows skipped because video file wasn't found at: D:/project/mcd_rppg_60_patients

Initializing Stage 1 Augmented Validation Dataset...
📦 Stage 1 Pipeline initialized:
   ↳ 72 samples successfully matched.
   ↳ 2880 rows filtered out (allocated to different split).
   ⚠️ Warning: 648 rows skipped because video file wasn't found at: D:/project/mcd_rppg_60_patients

🚀 Running Two-Stage Pipeline Optimization Engine (20 Epochs) on: cpu


AttributeError: module 'mediapipe' has no attribute 'solutions'